# FSAKE Checkpoint Forensics

This notebook audits Hugging Face checkpoints for a specific experiment and helps explain large validation-vs-test gaps.

What it does:
- Lists files for an experiment on HF.
- Downloads `checkpoint.pth.tar` and `model_best.pth.tar` (if present).
- Prints metadata (`iteration`, `val_acc`, `best_val_acc`) and weight fingerprints.
- Verifies whether `model_best` and `checkpoint` are actually different.
- Optionally runs `eval.py` on the downloaded checkpoints if the local FSAKE repo/data are available.

Security note:
- The token is read from environment variable or prompt at runtime.
- The token is not stored in this notebook file.

In [1]:
# Setup
import os
import json
import hashlib
from pathlib import Path
from getpass import getpass

import torch
from huggingface_hub import HfApi, hf_hub_download, login

print('Imports OK')

Imports OK


C:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Config
HF_REPO_ID = 'alkav/fsake-checkpoints'
EXP_NAME = 'D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222'  # change if needed

# Prefer environment variable; fallback to secure prompt
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    HF_TOKEN = getpass('Enter HF token (input hidden): ').strip()

if not HF_TOKEN:
    raise RuntimeError('No HF token provided.')

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()

AUDIT_DIR = Path('./hf_checkpoint_audit') / EXP_NAME
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', HF_REPO_ID)
print('Experiment:', EXP_NAME)
print('Audit dir:', AUDIT_DIR.resolve())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Repo: alkav/fsake-checkpoints
Experiment: D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222
Audit dir: E:\projects\few_shot_gnn\hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222


In [3]:
# List files for this experiment in HF
all_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type='model')
exp_files = sorted([f for f in all_files if f.startswith(EXP_NAME + '/')])

print(f'Total files in repo: {len(all_files)}')
print(f'Files under {EXP_NAME}:')
for f in exp_files:
    print(' -', f)

needed = [
    f'{EXP_NAME}/checkpoint.pth.tar',
    f'{EXP_NAME}/model_best.pth.tar',
    f'{EXP_NAME}/best_log.pth.tar',
]

print('\nPresence check:')
for f in needed:
    print(f'  {f}:', 'YES' if f in all_files else 'NO')

Total files in repo: 10
Files under D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222:
 - D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/checkpoint.pth.tar
 - D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/model_best.pth.tar

Presence check:
  D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/checkpoint.pth.tar: YES
  D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/model_best.pth.tar: YES
  D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/best_log.pth.tar: NO


In [4]:
# Download checkpoints (if they exist)
local_paths = {}
for name in ['checkpoint.pth.tar', 'model_best.pth.tar', 'best_log.pth.tar']:
    remote_path = f'{EXP_NAME}/{name}'
    try:
        src = hf_hub_download(repo_id=HF_REPO_ID, filename=remote_path, repo_type='model')
        dst = AUDIT_DIR / name
        Path(dst).write_bytes(Path(src).read_bytes())
        local_paths[name] = str(dst)
        print(f'OK: {name} -> {dst}')
    except Exception as e:
        print(f'MISSING: {name} ({e})')

print('\nDownloaded files:')
for k, v in local_paths.items():
    print(' -', k, '=>', v)

OK: checkpoint.pth.tar -> hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\checkpoint.pth.tar
OK: model_best.pth.tar -> hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\model_best.pth.tar


MISSING: best_log.pth.tar (404 Client Error. (Request ID: Root=1-69ec5c84-0f1368d20933ba17307275bb;bed0ba78-251d-4693-86be-2c9c0e87ee92)

Entry Not Found for url: https://huggingface.co/alkav/fsake-checkpoints/resolve/main/D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222/best_log.pth.tar.)

Downloaded files:
 - checkpoint.pth.tar => hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\checkpoint.pth.tar
 - model_best.pth.tar => hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\model_best.pth.tar


In [5]:
# Inspect checkpoint payloads and compare fingerprints
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def tensor_fingerprint(state_dict, max_tensors=8):
    # Lightweight deterministic fingerprint of the first tensors by key order
    keys = sorted(state_dict.keys())[:max_tensors]
    h = hashlib.sha256()
    for k in keys:
        t = state_dict[k].detach().cpu().contiguous()
        h.update(k.encode('utf-8'))
        h.update(str(tuple(t.shape)).encode('utf-8'))
        h.update(t.numpy().tobytes())
    return h.hexdigest(), keys

def summarize_ckpt(path):
    data = torch.load(path, map_location='cpu', weights_only=False)
    summary = {
        'path': path,
        'size_mb': round(Path(path).stat().st_size / (1024 * 1024), 2),
        'sha256': sha256_file(path),
        'keys': sorted(list(data.keys())),
        'iteration': data.get('iteration', None),
        'val_acc': data.get('val_acc', None),
        'best_val_acc': data.get('best_val_acc', None),
    }

    enc_sd = data.get('enc_module_state_dict', {})
    unet_sd = data.get('unet_module_state_dict', {})
    if enc_sd:
        summary['enc_fp'], summary['enc_fp_keys'] = tensor_fingerprint(enc_sd)
    if unet_sd:
        summary['unet_fp'], summary['unet_fp_keys'] = tensor_fingerprint(unet_sd)
    return summary

summaries = {}
for n in ['checkpoint.pth.tar', 'model_best.pth.tar']:
    if n in local_paths:
        summaries[n] = summarize_ckpt(local_paths[n])
        print('\n===', n, '===')
        for k in ['path', 'size_mb', 'sha256', 'iteration', 'val_acc', 'best_val_acc']:
            print(f'{k}:', summaries[n].get(k))

if 'checkpoint.pth.tar' in summaries and 'model_best.pth.tar' in summaries:
    same_file = summaries['checkpoint.pth.tar']['sha256'] == summaries['model_best.pth.tar']['sha256']
    same_enc = summaries['checkpoint.pth.tar'].get('enc_fp') == summaries['model_best.pth.tar'].get('enc_fp')
    same_unet = summaries['checkpoint.pth.tar'].get('unet_fp') == summaries['model_best.pth.tar'].get('unet_fp')

    print('\n=== checkpoint vs model_best comparison ===')
    print('Identical file bytes:', same_file)
    print('Same encoder weights fingerprint:', same_enc)
    print('Same UNet weights fingerprint:', same_unet)
    if same_file:
        print('WARNING: model_best and checkpoint are identical bytes.')
    elif same_enc and same_unet:
        print('WARNING: Files differ but effective model weights appear identical.')


=== checkpoint.pth.tar ===
path: hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\checkpoint.pth.tar
size_mb: 24.99
sha256: fbcc4e9c7524ea8549f1fe2e2f26b6e90f4ab3d41acb8dab6fcc6f31c6a34e40
iteration: 100000
val_acc: 0.8642000412940979
best_val_acc: 0.8798800432682037

=== model_best.pth.tar ===
path: hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\model_best.pth.tar
size_mb: 24.99
sha256: 7de01316cf5ed6944ca7c0b8faeec483c425423a16bcc2252cef818b6436582d
iteration: 95600
val_acc: 0.8798800432682037
best_val_acc: 0.8798800432682037

=== checkpoint vs model_best comparison ===
Identical file bytes: False
Same encoder weights fingerprint: False
Same UNet weights fingerprint: False


In [6]:
# Optional: recent repo commits (manual cross-check with HF UI)
commits = api.list_repo_commits(repo_id=HF_REPO_ID, repo_type='model')
print('Recent commits:')
for c in commits[:15]:
    print(f"- {c.commit_id[:8]} | {c.created_at} | {c.title}")

print('\nTip: Open the HF Files/History page and verify whether recent uploads for this EXP_NAME')
print('updated only checkpoint.pth.tar or also model_best.pth.tar.')

Recent commits:
- b5d4ae9b | 2026-04-23 15:52:50+00:00 | Upload D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222/checkpoint.pth.tar with huggingface_hub
- 703821fe | 2026-04-23 15:49:32+00:00 | Upload D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222/checkpoint.pth.tar with huggingface_hub
- 607ef29b | 2026-04-23 15:46:16+00:00 | Upload D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222/checkpoint.pth.tar with huggingface_hub
- 1fe2d6a5 | 2026-04-23 15:43:00+00:00 | Upload D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222/checkpoint.pth.tar with huggingface_hub
- cbf6b004 | 2026-04-23 15:39:45+00:00 | Upload D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222/checkpoint.pth.tar with huggingface_hub
- 2c5fa322 | 2026-04-23 15:36:32+00:00 | Upload D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222/checkpoint.pth.tar with huggingface_hub
- 4c6e

In [7]:
# Optional: run eval.py with explicit args if local FSAKE repo and dataset exist
import subprocess
import re
import sys

FSAKE_DIR_CANDIDATES = [
    Path('/content/FSAKE'),
    Path('E:/projects/few_shot_gnn/fsake_repo_clone'),
]

EVAL_ARGS_BASELINE = [
    '--dataset', 'mini',
    '--num_ways', '5',
    '--num_shots', '1',
    '--transductive', 'True',
    '--pool_mode', 'support',
    '--unet_mode', 'addold',
    '--seed', '222',
]

acc_re = re.compile(r"evaluation: total_count=\d+, accuracy: mean=([0-9.]+)%")

def run_eval_if_possible(extra_args=None):
    extra_args = extra_args or []
    fsake_dir = next((p for p in FSAKE_DIR_CANDIDATES if (p / 'eval.py').exists()), None)
    if fsake_dir is None:
        print('No local FSAKE repo found. Skipping eval run.')
        return None

    cmd = [sys.executable, 'eval.py'] + EVAL_ARGS_BASELINE + extra_args
    print('Running in', fsake_dir)
    print('Command:', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(fsake_dir), capture_output=True, text=True)
    out = proc.stdout + '\n' + proc.stderr
    print(out[-4000:])

    m = acc_re.search(out)
    acc = float(m.group(1)) if m else None
    print('Return code:', proc.returncode, '| Parsed mean acc:', acc)
    return {'rc': proc.returncode, 'acc': acc, 'log': out}

baseline_result = run_eval_if_possible([])
lmt_result = run_eval_if_possible(['--interaction_block', 'lmt', '--mediator_tokens', '16', '--mediator_layers', '3', '--mediator_heads', '8'])

Running in E:\projects\few_shot_gnn\fsake_repo_clone
Command: C:\Python314\python.exe eval.py --dataset mini --num_ways 5 --num_shots 1 --transductive True --pool_mode support --unet_mode addold --seed 222



Traceback (most recent call last):
  File "E:\projects\few_shot_gnn\fsake_repo_clone\eval.py", line 2, in <module>
    from data import MiniImagenetLoader,TieredImagenetLoader,Cub200Loader,CifarFsLoader
  File "E:\projects\few_shot_gnn\fsake_repo_clone\data.py", line 10, in <module>
    from torchvision import transforms
  File "C:\Python314\Lib\site-packages\torchvision\__init__.py", line 10, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torchvision\_meta_registrations.py", line 163, in <module>
    @torch.library.register_fake("torchvision::nms")
     ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torch\library.py", line 1073, in register
    use_lib._register_fake(
    ~~~~~~~~~~~~~~~~~~~~~~^
        op_name, func, _stacklevel=stacklevel +


Traceback (most recent call last):
  File "E:\projects\few_shot_gnn\fsake_repo_clone\eval.py", line 2, in <module>
    from data import MiniImagenetLoader,TieredImagenetLoader,Cub200Loader,CifarFsLoader
  File "E:\projects\few_shot_gnn\fsake_repo_clone\data.py", line 10, in <module>
    from torchvision import transforms
  File "C:\Python314\Lib\site-packages\torchvision\__init__.py", line 10, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torchvision\_meta_registrations.py", line 163, in <module>
    @torch.library.register_fake("torchvision::nms")
     ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torch\library.py", line 1073, in register
    use_lib._register_fake(
    ~~~~~~~~~~~~~~~~~~~~~~^
        op_name, func, _stacklevel=stacklevel +

In [8]:
# Final structured verdict scaffold
verdict = {
    'repo': HF_REPO_ID,
    'experiment': EXP_NAME,
    'has_checkpoint': 'checkpoint.pth.tar' in local_paths,
    'has_model_best': 'model_best.pth.tar' in local_paths,
    'checkpoint_iteration': summaries.get('checkpoint.pth.tar', {}).get('iteration'),
    'checkpoint_best_val_acc': summaries.get('checkpoint.pth.tar', {}).get('best_val_acc'),
    'model_best_iteration': summaries.get('model_best.pth.tar', {}).get('iteration'),
    'model_best_best_val_acc': summaries.get('model_best.pth.tar', {}).get('best_val_acc'),
    'checkpoint_sha256': summaries.get('checkpoint.pth.tar', {}).get('sha256'),
    'model_best_sha256': summaries.get('model_best.pth.tar', {}).get('sha256'),
}

print(json.dumps(verdict, indent=2))

if verdict['has_checkpoint'] and not verdict['has_model_best']:
    print('\nLikely issue: eval expects model_best.pth.tar but only checkpoint.pth.tar is present.')
elif verdict['has_checkpoint'] and verdict['has_model_best']:
    if verdict['checkpoint_sha256'] == verdict['model_best_sha256']:
        print('\nLikely issue: model_best and checkpoint are byte-identical; best-model selection may be ineffective or overwritten.')
    else:
        print('\nBoth checkpoint and model_best exist and differ. Next step: verify dataset/protocol consistency and run matched val/test from same loaded state.')
else:
    print('\nNo usable checkpoint found for this experiment path.')

{
  "repo": "alkav/fsake-checkpoints",
  "experiment": "D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222",
  "has_checkpoint": true,
  "has_model_best": true,
  "checkpoint_iteration": 100000,
  "checkpoint_best_val_acc": 0.8798800432682037,
  "model_best_iteration": 95600,
  "model_best_best_val_acc": 0.8798800432682037,
  "checkpoint_sha256": "fbcc4e9c7524ea8549f1fe2e2f26b6e90f4ab3d41acb8dab6fcc6f31c6a34e40",
  "model_best_sha256": "7de01316cf5ed6944ca7c0b8faeec483c425423a16bcc2252cef818b6436582d"
}

Both checkpoint and model_best exist and differ. Next step: verify dataset/protocol consistency and run matched val/test from same loaded state.


In [9]:
# Multi-experiment audit config (baseline + LMT)
EXPERIMENTS = [
    {
        'label': 'baseline',
        'exp_name': 'D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222',
        'eval_args': [
            '--dataset', 'mini',
            '--num_ways', '5',
            '--num_shots', '1',
            '--transductive', 'True',
            '--pool_mode', 'support',
            '--unet_mode', 'addold',
            '--seed', '222',
        ],
    },
    {
        'label': 'lmt',
        'exp_name': 'D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222',
        'eval_args': [
            '--dataset', 'mini',
            '--num_ways', '5',
            '--num_shots', '1',
            '--transductive', 'True',
            '--pool_mode', 'support',
            '--unet_mode', 'addold',
            '--seed', '222',
            '--interaction_block', 'lmt',
            '--mediator_tokens', '16',
            '--mediator_layers', '3',
            '--mediator_heads', '8',
        ],
    },
]

print('Configured experiments:')
for e in EXPERIMENTS:
    print('-', e['label'], '=>', e['exp_name'])

Configured experiments:
- baseline => D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222
- lmt => D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222


In [10]:
# Side-by-side audit for all configured experiments
from dataclasses import dataclass

@dataclass
class CkptSummary:
    exists: bool
    path: str = None
    size_mb: float = None
    sha256: str = None
    iteration: int = None
    val_acc: float = None
    best_val_acc: float = None


def download_if_exists(exp_name, filename):
    remote = f'{exp_name}/{filename}'
    try:
        src = hf_hub_download(repo_id=HF_REPO_ID, filename=remote, repo_type='model')
        dst_dir = Path('./hf_checkpoint_audit') / exp_name
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / filename
        Path(dst).write_bytes(Path(src).read_bytes())
        return str(dst)
    except Exception:
        return None


def summarize_path(path):
    if not path or not Path(path).exists():
        return CkptSummary(False)
    obj = torch.load(path, map_location='cpu', weights_only=False)
    return CkptSummary(
        exists=True,
        path=path,
        size_mb=round(Path(path).stat().st_size / (1024 * 1024), 2),
        sha256=sha256_file(path),
        iteration=obj.get('iteration'),
        val_acc=obj.get('val_acc'),
        best_val_acc=obj.get('best_val_acc'),
    )


multi_audit = {}
for exp in EXPERIMENTS:
    exp_name = exp['exp_name']
    ckpt_path = download_if_exists(exp_name, 'checkpoint.pth.tar')
    best_path = download_if_exists(exp_name, 'model_best.pth.tar')

    ckpt = summarize_path(ckpt_path)
    best = summarize_path(best_path)

    multi_audit[exp_name] = {
        'checkpoint': ckpt,
        'model_best': best,
    }


def fmt_float(v):
    return 'None' if v is None else f'{v:.6f}'

print('\n=== CHECKPOINT AUDIT TABLE ===')
print('label | file | exists | iter | val_acc | best_val_acc | size_mb | sha256_prefix')
print('-' * 120)
for exp in EXPERIMENTS:
    label = exp['label']
    exp_name = exp['exp_name']
    for fname in ['checkpoint', 'model_best']:
        s = multi_audit[exp_name][fname]
        sha_pref = (s.sha256[:12] if s.sha256 else 'None')
        print(f"{label:8} | {fname:10} | {str(s.exists):6} | {str(s.iteration):>6} | {fmt_float(s.val_acc):>10} | {fmt_float(s.best_val_acc):>12} | {str(s.size_mb):>7} | {sha_pref}")

print('\n=== INTRA-EXPERIMENT COMPARISON ===')
for exp in EXPERIMENTS:
    exp_name = exp['exp_name']
    c = multi_audit[exp_name]['checkpoint']
    b = multi_audit[exp_name]['model_best']
    print(f"\n{exp['label']} -> {exp_name}")
    if not c.exists or not b.exists:
        print('  Missing one of checkpoint/model_best')
        continue
    print('  same_bytes:', c.sha256 == b.sha256)
    print('  checkpoint_iter:', c.iteration, '| model_best_iter:', b.iteration)
    print('  checkpoint_best_val_acc:', c.best_val_acc, '| model_best_best_val_acc:', b.best_val_acc)


=== CHECKPOINT AUDIT TABLE ===
label | file | exists | iter | val_acc | best_val_acc | size_mb | sha256_prefix
------------------------------------------------------------------------------------------------------------------------
baseline | checkpoint | True   | 100000 |   0.864200 |     0.879880 |   24.99 | fbcc4e9c7524
baseline | model_best | True   |  95600 |   0.879880 |     0.879880 |   24.99 | 7de01316cf5e
lmt      | checkpoint | True   | 100000 |   0.984880 |     0.985600 |   30.71 | 49485763c999
lmt      | model_best | True   |  95200 |   0.985600 |     0.985600 |   30.71 | f0baf95c99c2

=== INTRA-EXPERIMENT COMPARISON ===

baseline -> D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222
  same_bytes: False
  checkpoint_iter: 100000 | model_best_iter: 95600
  checkpoint_best_val_acc: 0.8798800432682037 | model_best_best_val_acc: 0.8798800432682037

lmt -> D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222
  same_bytes: False
  checkpoint_iter: 

In [11]:
# Evaluation phase in this notebook (baseline + LMT)
# This cell runs eval.py for each configured experiment and parses mean accuracy.

import shutil
import subprocess
import re
import sys

ACC_RE = re.compile(r"evaluation: total_count=\d+, accuracy: mean=([0-9.]+)%, std=([0-9.]+)%, ci95=([0-9.]+)%")

FSAKE_DIR_CANDIDATES = [
    Path('/content/FSAKE'),
    Path('E:/projects/few_shot_gnn/fsake_repo_clone'),
]

fsake_dir = next((p for p in FSAKE_DIR_CANDIDATES if (p / 'eval.py').exists()), None)
if fsake_dir is None:
    raise RuntimeError('Could not find FSAKE repo with eval.py. Update FSAKE_DIR_CANDIDATES.')

print('Using FSAKE dir:', fsake_dir)
asset_ckpt_root = fsake_dir / 'asset' / 'checkpoints'
asset_ckpt_root.mkdir(parents=True, exist_ok=True)

# Ensure downloaded checkpoints are mirrored into FSAKE expected path.
for exp in EXPERIMENTS:
    exp_name = exp['exp_name']
    exp_dst = asset_ckpt_root / exp_name
    exp_dst.mkdir(parents=True, exist_ok=True)

    for filename in ['checkpoint.pth.tar', 'model_best.pth.tar']:
        src = Path('./hf_checkpoint_audit') / exp_name / filename
        if src.exists():
            dst = exp_dst / filename
            shutil.copy2(src, dst)
            print(f'Copied {src} -> {dst}')
        else:
            print(f'Missing local file for mirror: {src}')

results = []
for exp in EXPERIMENTS:
    label = exp['label']
    exp_name = exp['exp_name']
    cmd = [sys.executable, 'eval.py'] + exp['eval_args']
    print('\n' + '=' * 80)
    print('Running:', label)
    print('Command:', ' '.join(cmd))

    proc = subprocess.run(cmd, cwd=str(fsake_dir), capture_output=True, text=True)
    combined = proc.stdout + '\n' + proc.stderr
    print(combined[-5000:])

    m = ACC_RE.search(combined)
    mean = float(m.group(1)) if m else None
    std = float(m.group(2)) if m else None
    ci95 = float(m.group(3)) if m else None

    results.append({
        'label': label,
        'exp_name': exp_name,
        'return_code': proc.returncode,
        'mean_acc': mean,
        'std': std,
        'ci95': ci95,
    })

print('\n=== EVAL SUMMARY ===')
for r in results:
    print(f"{r['label']:8} | rc={r['return_code']} | mean={r['mean_acc']} | std={r['std']} | ci95={r['ci95']}")

# Convenience delta if both succeeded
base = next((r for r in results if r['label'] == 'baseline' and r['mean_acc'] is not None), None)
lmt = next((r for r in results if r['label'] == 'lmt' and r['mean_acc'] is not None), None)
if base and lmt:
    print(f"\nDelta (LMT - Baseline): {lmt['mean_acc'] - base['mean_acc']:+.2f} percentage points")

Using FSAKE dir: E:\projects\few_shot_gnn\fsake_repo_clone
Copied hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\checkpoint.pth.tar -> E:\projects\few_shot_gnn\fsake_repo_clone\asset\checkpoints\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\checkpoint.pth.tar
Copied hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\model_best.pth.tar -> E:\projects\few_shot_gnn\fsake_repo_clone\asset\checkpoints\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222\model_best.pth.tar
Copied hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222\checkpoint.pth.tar -> E:\projects\few_shot_gnn\fsake_repo_clone\asset\checkpoints\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222\checkpoint.pth.tar
Copied hf_checkpoint_audit\D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222\model_best.pth.tar -> E:\projects\few_shot_gnn\fsake_repo_clone\a


Traceback (most recent call last):
  File "E:\projects\few_shot_gnn\fsake_repo_clone\eval.py", line 2, in <module>
    from data import MiniImagenetLoader,TieredImagenetLoader,Cub200Loader,CifarFsLoader
  File "E:\projects\few_shot_gnn\fsake_repo_clone\data.py", line 10, in <module>
    from torchvision import transforms
  File "C:\Python314\Lib\site-packages\torchvision\__init__.py", line 10, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torchvision\_meta_registrations.py", line 163, in <module>
    @torch.library.register_fake("torchvision::nms")
     ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torch\library.py", line 1073, in register
    use_lib._register_fake(
    ~~~~~~~~~~~~~~~~~~~~~~^
        op_name, func, _stacklevel=stacklevel +


Traceback (most recent call last):
  File "E:\projects\few_shot_gnn\fsake_repo_clone\eval.py", line 2, in <module>
    from data import MiniImagenetLoader,TieredImagenetLoader,Cub200Loader,CifarFsLoader
  File "E:\projects\few_shot_gnn\fsake_repo_clone\data.py", line 10, in <module>
    from torchvision import transforms
  File "C:\Python314\Lib\site-packages\torchvision\__init__.py", line 10, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torchvision\_meta_registrations.py", line 163, in <module>
    @torch.library.register_fake("torchvision::nms")
     ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\site-packages\torch\library.py", line 1073, in register
    use_lib._register_fake(
    ~~~~~~~~~~~~~~~~~~~~~~^
        op_name, func, _stacklevel=stacklevel +